In [51]:
import os, json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

os.environ["HF"] = os.getenv("HF")
llm = ChatGroq(model=os.getenv("GROQ_MODEL"), api_key=os.getenv("GROQ_API_KEY"))
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9870.75it/s]


In [8]:
from pathlib import Path
data_folder = Path("../src/data")
transactions_file_path = data_folder / "transactions.json"
print(transactions_file_path.exists())

True


In [ ]:
# Load and format transactions docs
def format_transactions_metadata(record: dict, metadata: dict) -> dict:
    metadata["id"]= record.get("id")
    metadata["status"] = record.get("status")
    metadata["type"] = record.get("type")
    metadata["amount"] = record.get("amount")
    metadata["name"] = record.get("name")
    return metadata

# create the JSON data loader
loader = JSONLoader(
    file_path=transactions_file_path,
    jq_schema=".[]",
    text_content=False,
    metadata_func=format_transactions_metadata
)

# page_content is loaded as a raw JSON string and after document loading, it is converted to human-readable text for the embedding model 
# to create embeddings as the embedding model used here is trained on natural language sentences.

# load the documents
docs = loader.load()

In [13]:
print(f"Example metadata : {docs[0].metadata} \n------\nExample page_content: {docs[0].page_content}")

Example metadata : {'source': '/Users/prakhardhama/Desktop/Agentic AI/projects/personal-transaction-assistant/src/data/transactions.json', 'seq_num': 1, 'id': 'TXN20260301001', 'status': 'success', 'type': 'debit', 'amount': 2500, 'name': 'Rajesh Kumar'} 
------
Example page_content: {"id": "TXN20260301001", "name": "Rajesh Kumar", "upiId": "rajesh.k@okicici", "avatar": "RK", "amount": 2500, "type": "debit", "date": "2026-03-28T18:45:00", "status": "success", "paymentMethod": "UPI", "note": "Rent March 2026", "bankRef": "609284710293", "fromAccount": "HDFC Bank \u2022\u20221847"}


In [15]:
# Re-writing page_content to human-readable text
for doc in docs:
    data = json.loads(doc.page_content)
    transaction_direction = "Sent to" if data.get("type") == "debit" else "Recieved from"
    doc.page_content = (
        f"Transaction ID: {data.get('id')} | "
        f"{transaction_direction} {data.get('name')} ({data.get('upiId')}) | "
        f"Amount: ₹{data.get('amount')} | "
        f"Status: {data.get('status')} | "
        f"Method: {data.get('paymentMethod')} | "
        f"Date: {data.get('date')} | "
        f"Note: {data.get('note')} | "
        f"Bank Ref: {data.get('bankRef')}"
    )

In [46]:
# from langchain.retrievers.self_query.base import SelfQueryRetriever
# from langchain_classic.chains.query_constructor.schema import AttributeInfo

# metadata_field_info = [
#     AttributeInfo(
#         name="name",
#         description="Name of the person or merchant the transaction was made with",
#         type="string"
#     ),
#     AttributeInfo(
#         name="type",
#         description="Transaction type - either 'debit' (money sent) or 'credit' (money received)",
#         type="string"
#     ),
#     AttributeInfo(
#         name="status",
#         description="Transaction status - either 'success', 'failed', or 'pending'",
#         type="string"
#     ),
#     AttributeInfo(
#         name="amount",
#         description="Transaction amount in Indian Rupees",
#         type="float"
#     ),
#     AttributeInfo(
#         name="id",
#         description="Unique transaction ID like TXN20260301001",
#         type="string"
#     ),
# ]

vectorstore = FAISS.from_documents(docs, embedding)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 20, "fetch_k": 30}
)


# retriever = SelfQueryRetriever.from_llm(
#     llm=llm,
#     vectorstore=vectorstore,
#     document_contents="Personal UPI payment transactions of a user",
#     metadata_field_info=metadata_field_info,
#     verbose=True,
#     enable_limit=True
# )

some dependency issue with SelfQueryRetriever

In [47]:
# Prompt and Chain
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are PersonalPayBot, a personal transaction assistant for a PhonePe-style payments app.
                Answer **ONLY** from the **CONTEXT** provided. Do not guess or hallucinate.
                Use ₹ symbol for amounts. Keep answers short and clear. If you don't find any related answer, do not hallucinate 
                and simply tell that you can not find the transaction required by the user.
     
                <context>
                {context}
                </context>"""),

    ("human", "{input}")
])

In [48]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
retriever_chain = create_retrieval_chain(retriever, document_chain)

In [49]:
response = retriever_chain.invoke({"input": "What is the total amount i sent to Amit?"})
print(response['answer'])

You sent ₹5000 (failed), ₹2750 (success) and ₹3200 (success) to Amit Patel. 
The total successful amount is ₹5950 (₹2750 + ₹3200).


In [50]:
response = retriever_chain.invoke({"input": "What is the total amount i sent to Prakhar?"})
print(response['answer'])

I cannot find any transaction related to Prakhar.
